# Validation Run — Production Pipeline

**Scope:** Task 3.2 — Honest Validation Strategy

This notebook runs the **full competition-grade validation pipeline**
using the real `UnifiedFloodModel` (Tier 2 HeteroGNN-GRU), the
`ValidationRunner`, and the exact hierarchical SRMSE metric.

## Competition Scoring Formula

$$
\text{SRMSE} = \text{Mean}_{\text{models}} \left(
  \text{Mean}_{\text{events}} \left(
    \text{Mean}_{\text{node types}} \left(
      \text{Mean}_{\text{nodes}} \left(
        \frac{\text{RMSE}_i}{\sigma_i}
      \right)
    \right)
  \right)
\right)
$$

## Protocol

| Phase | Timesteps | Input | Scored? |
|-------|-----------|-------|---------|
| **Spin-up** | `t = 0 … 9` | Ground-truth (teacher forcing) | No |
| **Prediction** | `t = 10 … T-1` | Model's own output (autoregressive) | **Yes** |

## Notebook Sections

1. **Setup** — Imports, device, auto-reload
2. **Data Pipeline** — Load `FloodDataset`, compute per-node σ, build graphs
3. **Model** — Construct `UnifiedFloodModel` from data (or load checkpoint)
4. **Single-Event Validation** — Detailed rollout + scoring on one event
5. **Full Cross-Validation** — `ValidationRunner` with LOEO folds
6. **Diagnostics & Visualisation** — Per-node breakdown, prediction plots, submission sanity check

In [ ]:
# ── Cell 1: Setup & Imports ───────────────────────────────────────────
%load_ext autoreload
%autoreload 2

import sys, os, json, time, warnings
from pathlib import Path

# Ensure src/ is importable
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch import Tensor

# ── Project imports ───────────────────────────────────────────────────
from src.config import RAW_DATA_PATH
from src.dataset import FloodDataset
from src.graph_builder_unified import (
    build_unified_graph,
    get_feature_dims,
    summarise_graph,
)
from src.model_unified import UnifiedFloodModel
from src.loss import (
    FloodLoss,
    SRMSEAccumulator,
    standardized_rmse_metric,
    per_node_loss_breakdown,
)
from src.validate import (
    ValidationResult,
    ValidationRunner,
    validate_event_unified,
    extract_predictions,
)
from src.inference import SubmissionGenerator, predict_event

# ── Device ────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# ── Constants ─────────────────────────────────────────────────────────
SPINUP_STEPS = 10          # Competition spec: ground truth for t=0..9
MODEL_IDS    = ["1", "2"]  # Both urban models
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"RAW_DATA_PATH : {RAW_DATA_PATH}")
print(f"PyTorch       : {torch.__version__}")
print(f"Device        : {DEVICE}")
print(f"CUDA avail.   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory    : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Data Pipeline & Graph Construction

Load training data with `FloodDataset`, compute per-node standard
deviations (σ), and pre-build heterogeneous graphs.

**Key checks:**
- Static caching works (same object identity for same model).
- Node counts match between static and dynamic files.
- Per-node σ distribution is healthy (no near-zero values dominating).

In [ ]:
# ── Cell 2: Data Pipeline & Graph Construction ───────────────────────

dataset = FloodDataset(root_dir=str(RAW_DATA_PATH), mode="train")
print(f"Total training events: {len(dataset)}\n")

# ── Per-model summary ────────────────────────────────────────────────
stds_cache = {}   # model_id -> {"1d": np.array, "2d": np.array}
sample_graphs = {}  # model_id -> HeteroData (first event, for model init)

for mid in MODEL_IDS:
    model_ds = dataset.filter_by_model(mid)
    event_ids = model_ds.get_event_ids()
    print(f"Model {mid}: {len(model_ds)} events  (IDs: {event_ids[:5]}{'…' if len(event_ids) > 5 else ''})")

    # Fetch first event for inspection
    sample = model_ds[0]
    print(f"  Model ID : {sample['model_id']}")
    print(f"  Event ID : {sample['event_id']}")

    # Shape report
    for key in ["static_1d_nodes", "static_2d_nodes",
                "dynamic_1d_nodes", "dynamic_2d_nodes",
                "edge_index_1d", "edge_index_2d", "1d2d_conn"]:
        df = sample[key]
        print(f"    {key:25s} → {df.shape}")

    # Compute per-node stds
    stds = dataset.compute_node_stds(model_id=mid)
    model_stds = stds.get(mid, {"1d": np.array([1.0]), "2d": np.array([1.0])})
    stds_cache[mid] = model_stds

    stds_1d = model_stds["1d"]
    stds_2d = model_stds["2d"]
    print(f"\n  1D σ stats: min={stds_1d.min():.6f}, mean={stds_1d.mean():.6f}, max={stds_1d.max():.6f}")
    print(f"  2D σ stats: min={stds_2d.min():.6f}, mean={stds_2d.mean():.6f}, max={stds_2d.max():.6f}")
    print(f"  1D nodes with σ ≈ 0 (<1e-4): {(stds_1d < 1e-4).sum()} / {len(stds_1d)}")
    print(f"  2D nodes with σ ≈ 0 (<1e-4): {(stds_2d < 1e-4).sum()} / {len(stds_2d)}")

    # Build a sample graph (for model architecture init)
    graph = build_unified_graph(sample)
    sample_graphs[mid] = graph
    print(f"\n{summarise_graph(graph)}")

    # Cache verification
    if len(model_ds) > 1:
        s2 = model_ds[1]
        if s2["model_id"] == sample["model_id"]:
            assert s2["static_1d_nodes"] is sample["static_1d_nodes"], \
                "Cache miss — static DataFrames should share identity!"
            print("  ✓ Static cache verified (same object identity)")
    print()

print("✅ Data pipeline verification passed.")

## Model Construction

Two options:

1. **From scratch** — `UnifiedFloodModel.from_graph()` infers all
   input dimensions from a sample graph.  Uses un-trained weights.
2. **From checkpoint** — load a trained checkpoint (`.pt` file) for
   actual scoring.

Set `CHECKPOINT_PATH` below to switch.  When `None`, the model is
built fresh (useful for pipeline smoke-testing before training).

In [ ]:
# ── Cell 3: Model Construction ────────────────────────────────────────

# ── Configuration ────────────────────────────────────────────────────
CHECKPOINT_PATH = None  # Set to e.g. "checkpoints/best_model.pt" to load trained weights
HIDDEN_CHANNELS = 64
NUM_GNN_LAYERS  = 3
NUM_GRU_LAYERS  = 1
DROPOUT         = 0.1

# ── Build model ──────────────────────────────────────────────────────
if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    # ── Load from checkpoint ─────────────────────────────────────
    print(f"Loading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

    # Reconstruct architecture from saved config + sample graph dims
    ckpt_cfg = ckpt.get("config", {})
    ref_graph = list(sample_graphs.values())[0]
    dims = get_feature_dims(ref_graph)

    model = UnifiedFloodModel(
        in_channels_1d_static=dims["in_channels_1d_static"],
        in_channels_1d_dynamic=dims["in_channels_1d_dynamic"],
        in_channels_2d_static=dims["in_channels_2d_static"],
        in_channels_2d_dynamic=dims["in_channels_2d_dynamic"],
        hidden_channels=ckpt_cfg.get("hidden_channels", HIDDEN_CHANNELS),
        num_gnn_layers=ckpt_cfg.get("num_gnn_layers", NUM_GNN_LAYERS),
        num_gru_layers=ckpt_cfg.get("num_gru_layers", NUM_GRU_LAYERS),
        dropout=ckpt_cfg.get("dropout", DROPOUT),
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    epoch = ckpt.get("epoch", "?")
    best_srmse = ckpt.get("best_val_srmse", "?")
    print(f"  Loaded epoch {epoch}, best val SRMSE={best_srmse}")

else:
    # ── Build fresh (untrained) model ────────────────────────────
    ref_graph = list(sample_graphs.values())[0]
    model = UnifiedFloodModel.from_graph(
        ref_graph,
        hidden_channels=HIDDEN_CHANNELS,
        num_gnn_layers=NUM_GNN_LAYERS,
        num_gru_layers=NUM_GRU_LAYERS,
        dropout=DROPOUT,
    )
    model.to(DEVICE)
    print("  ⚠ Using untrained model — scores are meaningless but pipeline is verified.")

print(f"\n{model.summarise()}")

# Quick forward-pass smoke test
ref_graph_dev = ref_graph.to(DEVICE)
with torch.no_grad():
    model.eval()
    p1d, p2d = model.rollout(ref_graph_dev, spinup_steps=2, teacher_forcing_ratio=0.0)
    print(f"Smoke test: rollout → preds_1d={list(p1d.shape)}, preds_2d={list(p2d.shape)}")
    assert not torch.isnan(p1d).any(), "NaN in 1D predictions!"
    assert not torch.isnan(p2d).any(), "NaN in 2D predictions!"

print("✅ Model construction + smoke test passed.")

## Single-Event Validation (Detailed)

Runs the exact competition protocol on **one event** with full
diagnostics:

1. Spin-up (t=0..9): Ground truth → GRU hidden state warm-up.
2. Autoregressive prediction (t=10..T): Model-only, scored.
3. SRMSE computed per node type (1D, 2D) and combined.
4. Per-node loss breakdown to identify problem nodes.

This cell shows the detailed internals; the next cell automates this
across all events via `ValidationRunner`.

In [ ]:
# ── Cell 4: Single-Event Validation ──────────────────────────────────

# Pick one event from Model_1 for detailed analysis
VAL_MODEL_ID = "1"
model_ds = dataset.filter_by_model(VAL_MODEL_ID)
available = model_ds.get_event_ids()
val_eid = available[-1]  # Use last event as hold-out
print(f"Single-event validation: Model_{VAL_MODEL_ID}, Event_{val_eid}")
print(f"  Available events: {len(available)}")

# ── Build graph for this event ───────────────────────────────────────
_, val_ds = model_ds.split_by_event(val_eid)
sample = val_ds[0]
graph = build_unified_graph(sample)
print(f"  Timesteps: {graph.num_timesteps}")
print(f"  1D Nodes:  {graph['node_1d'].num_nodes}")
print(f"  2D Nodes:  {graph['node_2d'].num_nodes}")

# ── Retrieve per-node stds ───────────────────────────────────────────
stds_1d = torch.from_numpy(stds_cache[VAL_MODEL_ID]["1d"]).float()
stds_2d = torch.from_numpy(stds_cache[VAL_MODEL_ID]["2d"]).float()

# ── Run validation with full diagnostics ─────────────────────────────
result = validate_event_unified(
    model=model,
    data=graph,
    stds_1d=stds_1d,
    stds_2d=stds_2d,
    device=DEVICE,
    spinup_steps=SPINUP_STEPS,
    run_diagnostics=True,
    top_k=10,
)

print(f"\n{'='*50}")
print(f"  SRMSE  1D      : {result['srmse_1d']:.6f}")
print(f"  SRMSE  2D      : {result['srmse_2d']:.6f}")
print(f"  SRMSE combined : {result['srmse_combined']:.6f}")
print(f"  Elapsed        : {result['elapsed']:.2f}s")
print(f"{'='*50}")

# ── Diagnostics: worst-performing nodes ──────────────────────────────
if "diagnostics_1d" in result:
    d1d = result["diagnostics_1d"]
    print(f"\n  Top-10 worst 1D nodes (SRMSE):")
    for i, (idx, val) in enumerate(zip(d1d["top_k_indices"], d1d["top_k_srmse"])):
        print(f"    {i+1:2d}. Node {idx:4d} → SRMSE = {val:.4f}")

if "diagnostics_2d" in result:
    d2d = result["diagnostics_2d"]
    print(f"\n  Top-10 worst 2D nodes (SRMSE):")
    for i, (idx, val) in enumerate(zip(d2d["top_k_indices"][:5], d2d["top_k_srmse"][:5])):
        print(f"    {i+1:2d}. Node {idx:4d} → SRMSE = {val:.4f}")
    if len(d2d["top_k_indices"]) > 5:
        print(f"    ... ({len(d2d['top_k_indices']) - 5} more)")

# ── Prediction shape verification ────────────────────────────────────
preds_1d = result["preds_1d"]
preds_2d = result["preds_2d"]
print(f"\n  Predictions: 1D={list(preds_1d.shape)}, 2D={list(preds_2d.shape)}")
assert preds_1d.shape[0] == graph.num_timesteps
assert preds_2d.shape[0] == graph.num_timesteps
assert not torch.isnan(preds_1d).any(), "NaN in 1D predictions!"
assert not torch.isnan(preds_2d).any(), "NaN in 2D predictions!"

print("\n✅ Single-event validation passed.")

## Full Cross-Validation (Leave-One-Event-Out)

Uses `ValidationRunner` to run proper LOEO cross-validation across
both urban models.

**Key outputs:**
- Overall hierarchical SRMSE (the number we aim to minimise).
- Per-model, per-event, per-node-type breakdown.
- Per-node diagnostics for targeted model improvement.

> **Note:** With an untrained model, the SRMSE will be very high.
> After training, re-run to get meaningful scores.  Target SRMSE < 1.0
> for a competitive submission.

In [ ]:
# ── Cell 5: Full Cross-Validation ─────────────────────────────────────

# ── Option A: Single hold-out validation (fast) ─────────────────────
runner = ValidationRunner(
    model=model,
    data_root=str(RAW_DATA_PATH),
    device=DEVICE,
    model_ids=MODEL_IDS,
    spinup_steps=SPINUP_STEPS,
    run_diagnostics=True,
    verbose=True,
)

# Use the last available event as hold-out
holdout_result = runner.validate_holdout(val_event_id="4")

print("\n" + holdout_result.summary_str())

# ── Save results to disk ─────────────────────────────────────────────
results_dir = Path("..") / "results"
results_dir.mkdir(exist_ok=True)
holdout_result.save(results_dir / "validation_holdout.json")
print(f"Results saved to {results_dir / 'validation_holdout.json'}")

# ── Option B: Multi-fold cross-validation (thorough, slower) ─────────
# Uncomment below for a more robust score estimate:
#
# cv_results = runner.cross_validate(n_folds=5)
# print(f"\nCV Results: {cv_results['mean_srmse']:.6f} ± {cv_results['std_srmse']:.6f}")
# for i, score in enumerate(cv_results["per_fold_srmse"]):
#     print(f"  Fold {i+1}: {score:.6f} (event={cv_results['fold_events'][i]})")

## Diagnostics & Visualisation

Three visualisation panels:

1. **Prediction vs Ground Truth** — time series for best/worst nodes.
2. **Per-Node SRMSE Heatmap** — spatial overview of model accuracy.
3. **Submission Format Sanity Check** — verify the output matches
   competition requirements before submitting.

In [ ]:
# ── Cell 6: Diagnostics & Visualisation ──────────────────────────────

# ---- PANEL 1: Prediction vs Ground Truth time series -----------------

# Extract full rollout from the single-event validation
preds_dict = extract_predictions(model, graph, DEVICE, spinup_steps=SPINUP_STEPS)

T = preds_dict["num_timesteps"]
skip = preds_dict["spinup_steps"]
time_all = np.arange(T)
time_pred = np.arange(skip, T)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle(
    f"Prediction vs Ground Truth — Model {VAL_MODEL_ID}, Event {val_eid}",
    fontsize=13, fontweight="bold",
)

for row, (nt_label, preds_key, targets_key, stds_t) in enumerate([
    ("1D (Pipes)", "preds_1d", "targets_1d", stds_1d),
    ("2D (Surface)", "preds_2d", "targets_2d", stds_2d),
]):
    preds = preds_dict[preds_key]
    targets = preds_dict[targets_key]
    N = preds.shape[1]

    # SRMSE per node for this type
    per_node_rmse = torch.sqrt(((preds[skip:] - targets[skip:]) ** 2).mean(dim=0))
    per_node_srmse = per_node_rmse / stds_t.clamp(min=1e-6)

    best_node = int(per_node_srmse.argmin())
    worst_node = int(per_node_srmse.argmax())

    for col, (node_idx, node_label) in enumerate([
        (best_node, f"Best (node {best_node}, SRMSE={per_node_srmse[best_node]:.3f})"),
        (worst_node, f"Worst (node {worst_node}, SRMSE={per_node_srmse[worst_node]:.3f})"),
    ]):
        ax = axes[row, col]
        gt_node = targets[:, node_idx].numpy()
        pred_node = preds[:, node_idx].numpy()

        ax.plot(time_all, gt_node, label="Ground Truth", color="steelblue", lw=1.5)
        ax.plot(time_pred, pred_node[skip:], label="Prediction", color="tomato",
                lw=1.5, ls="--")
        ax.axvline(skip, color="grey", ls=":", lw=0.8)
        ax.axvspan(0, skip, alpha=0.05, color="grey")
        ax.set_title(f"{nt_label} — {node_label}", fontsize=10)
        ax.set_xlabel("Timestep")
        ax.set_ylabel("Water Level")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.2)

fig.tight_layout()
plt.show()

# ---- PANEL 2: Per-Node SRMSE Distribution ---------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Per-Node SRMSE Distribution", fontsize=13, fontweight="bold")

for i, (nt_label, preds_key, targets_key, stds_t) in enumerate([
    ("1D Nodes", "preds_1d", "targets_1d", stds_1d),
    ("2D Nodes", "preds_2d", "targets_2d", stds_2d),
]):
    preds = preds_dict[preds_key]
    targets = preds_dict[targets_key]
    per_node_rmse = torch.sqrt(((preds[skip:] - targets[skip:]) ** 2).mean(dim=0))
    per_node_srmse = (per_node_rmse / stds_t.clamp(min=1e-6)).numpy()

    ax = axes[i]
    ax.hist(per_node_srmse, bins=min(50, len(per_node_srmse)), color="steelblue",
            edgecolor="white", alpha=0.8)
    ax.axvline(np.mean(per_node_srmse), color="tomato", ls="--", lw=1.5,
               label=f"Mean = {np.mean(per_node_srmse):.3f}")
    ax.axvline(np.median(per_node_srmse), color="orange", ls=":", lw=1.5,
               label=f"Median = {np.median(per_node_srmse):.3f}")
    ax.set_title(nt_label, fontsize=10)
    ax.set_xlabel("SRMSE")
    ax.set_ylabel("Node Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2, axis="y")

fig.tight_layout()
plt.show()

# ---- PANEL 3: Submission Format Sanity Check -------------------------

print("\n" + "=" * 60)
print("  SUBMISSION FORMAT SANITY CHECK")
print("=" * 60)

# Build submission rows from this one event's predictions
from src.inference import _build_submission_rows_vectorized

df_1d = _build_submission_rows_vectorized(
    preds_dict["preds_1d"], VAL_MODEL_ID, val_eid, "1d", SPINUP_STEPS
)
df_2d = _build_submission_rows_vectorized(
    preds_dict["preds_2d"], VAL_MODEL_ID, val_eid, "2d", SPINUP_STEPS
)
df_sample = pd.concat([df_1d, df_2d], ignore_index=True)

print(f"  Submission rows (this event): {len(df_sample):,}")
print(f"  Columns: {list(df_sample.columns)}")
print(f"  NaN values: {df_sample['water_level'].isna().sum()}")
print(f"  Duplicates: {df_sample['row_id'].duplicated().sum()}")
print(f"  row_id sample: {df_sample['row_id'].iloc[0]}")
print(f"\n  First 5 rows:")
print(df_sample.head().to_string(index=False))

# Expected row count: (T - spinup) * N_nodes per type
T_scored = T - SPINUP_STEPS
n_1d = preds_dict["preds_1d"].shape[1]
n_2d = preds_dict["preds_2d"].shape[1]
expected_rows = T_scored * (n_1d + n_2d)
actual_rows = len(df_sample)
print(f"\n  Expected rows: {expected_rows:,}")
print(f"  Actual rows:   {actual_rows:,}")
assert actual_rows == expected_rows, f"Row count mismatch! {actual_rows} != {expected_rows}"

print("\n✅ All validation checks passed.")